# Optimización de base de datos para AireComprimido EC SAS

## Importación

In [167]:
import pandas as pd

base_df = pd.read_excel("../docs/EMCOVELE 2025.xlsx")
result_df = pd.DataFrame(columns=['PRODUCTO', 'MODELO', 'CANTIDAD_2025', 'FOB_UNIT_2025', 'CIF_UNIT_2025'])
prev_df = pd.read_excel('../docs/EMCOVELE_new.xlsx')

base_df.columns

Index(['Año', 'Mes', 'Distrito', 'Régimen', 'Importador', 'RUC', 'Provincia',
       'País_de_Origen', 'País_de_Embarque', 'Ciudad_de_Embarque',
       'Remitente/Proveedor', 'Manifiesto', 'BL', 'Transportista',
       'Consignatario', 'Consolidadora', 'Vía', 'Fecha_de_Embarque',
       'Fecha_de_Llegada', 'Incoterm', 'Subpartida', 'TNAN',
       'Descripción_Arancelaria', 'Marca_Comercial', 'Descripción_Comercial',
       'Producto', 'Modelo', 'Cantidad', 'FOB Total',
       'FOB_Unitario_(Calculado)', 'Flete', 'Seguro', 'CIF',
       'CFR_Unitario_(Calculado)', 'CIF_Unitario_(Calculado)',
       'Fodinfa_(Estimado)', 'Liberación', 'Peso_Neto_(Kg)', 'CIF UNITARIO',
       'FOB ATLAS', 'FOB CLEINTE', 'PVP ATLAS', 'FOB Atlas', 'FOB Cliente|',
       'PVP Atlas '],
      dtype='object')

## Filtro de columnas base

In [168]:
# base_filter_df = base_df[['DESCRIPCION PRODUCTO COMERCIAL', 'MODELO MERCADERIA', 'CANTIDAD', 'US$ FOB UNIT', 'US$ CIF']]
base_filter_df = base_df[['Descripción_Comercial', 'Modelo', 'Cantidad', 'FOB_Unitario_(Calculado)', 'CIF_Unitario_(Calculado)']]
base_filter_df = base_filter_df.sort_values(by=['Modelo', 'FOB_Unitario_(Calculado)'], ascending=[True, True])

# base_filter_df = base_filter_df.rename(columns={'Descripción_Comercial' : 'PRODUCTO','MODELO MERCADERIA':'MODELO', 'FOB_Unitario_(Calculado)':'FOB_UNIT_2025', 'Cantidad':'CANTIDAD_2025'})
base_filter_df = base_filter_df.rename(columns={'Descripción_Comercial' : 'PRODUCTO','MODELO MERCADERIA':'MODELO', 'FOB_Unitario_(Calculado)':'FOB_UNIT_2025', 'Cantidad':'CANTIDAD_2025', 'CIF_Unitario_(Calculado)':'CIF_UNIT_2025', 'Modelo':'MODELO'})

# base_filter_df['CIF_UNIT_2025'] = base_filter_df['US$ CIF'] / base_filter_df['CANTIDAD_2025']

# base_filter_df = base_filter_df.drop(columns=['US$ CIF'])
base_filter_df['MODELO'] = base_filter_df['MODELO'].str.strip("'")
base_filter_df = base_filter_df.iloc[:-1]
base_filter_df


,PRODUCTO,MODELO,CANTIDAD_2025,FOB_UNIT_2025,CIF_UNIT_2025
1,SCREW COMPRESSOR,0000-0087-13,1,283.84,337.98
2,SCREW COMPRESSOR,0000-0087-24,1,157.66,187.74
3,SCREW COMPRESSOR,0000-0087-49,1,201.95,240.47
4,SCREW COMPRESSOR,0000-0092-45,1,267.72,289.99
20,SCREW COMPRESSOR,0000-0505-24,1,111.14,135.65
...,...,...,...,...,...
1545,COMPRESOR,8153611770,1,8600,9031.2
1567,WASHER,9125-6303-00,1,1.9,2.78
1565,WASHER,9125-6303-00,1,1.91,2.6
1566,WASHER,9125-6303-00,1,1.91,2.6


## Reduccion tabla filtrada

In [169]:
def clean_df(df):
  # Convertimos a listas para manipular valores y cantidades
  precios = df['FOB_UNIT_2025'].tolist()
  cantidades = df['CANTIDAD_2025'].tolist()
  indices = df.index.tolist()
  
  idx_evaluar = len(precios) - 1
  
  while idx_evaluar > 0:
    precio_actual = precios[idx_evaluar]
    fusionado = False
      
    for i in range(idx_evaluar):
      precio_referencia = precios[i]
      
      # Calculamos la diferencia
      diferencia = (precio_actual - precio_referencia) / precio_actual
      
      if diferencia < 0.12:
        while(i < idx_evaluar):
          cantidades[i] += cantidades[idx_evaluar]
          
          # Eliminamos el valor caro de nuestras listas
          precios.pop(idx_evaluar)
          cantidades.pop(idx_evaluar)
          indices.pop(idx_evaluar)
          idx_evaluar -= 1
        fusionado = True
        break
    if(not fusionado):
      idx_evaluar -= 1
  
  # Reconstruimos el DataFrame con las cantidades actualizadas
  resultado = df.loc[indices].copy()
  resultado['CANTIDAD_2025'] = cantidades
  return resultado

model_list = base_filter_df['MODELO'].unique()

for model in model_list:
  df_by_model = base_filter_df[base_filter_df['MODELO'] == model]
  df_clean = clean_df(df_by_model)
  result_df = pd.concat([result_df, df_clean], ignore_index=True)

result_df


,PRODUCTO,MODELO,CANTIDAD_2025,FOB_UNIT_2025,CIF_UNIT_2025
0,SCREW COMPRESSOR,0000-0087-13,1,283.84,337.98
1,SCREW COMPRESSOR,0000-0087-24,1,157.66,187.74
2,SCREW COMPRESSOR,0000-0087-49,1,201.95,240.47
3,SCREW COMPRESSOR,0000-0092-45,1,267.72,289.99
4,SCREW COMPRESSOR,0000-0505-24,1,111.14,135.65
...,...,...,...,...,...
965,SCREW COMPRESSOR,8153-9182-54,1,15359.27,18037.16
966,COMPRESOR,8153038222,1,13100,13531.2
967,COMPRESOR,8153611770,1,8600,9031.2
968,WASHER,9125-6303-00,3,1.9,2.78


## Juntar tablas

In [170]:

prev_df['REPS'] = prev_df.groupby('MODELO').cumcount()
result_df['REPS'] = result_df.groupby('MODELO').cumcount()

df_final = pd.merge(prev_df, result_df, on=['MODELO', 'REPS'], how='outer')


df_final['PRODUCTO'] = df_final['PRODUCTO_x'].combine_first(df_final['PRODUCTO_y'])


df_final['FOB_UNIT_2023'] = df_final['FOB_UNIT_2023'].fillna(0)
df_final['FOB_UNIT_2024'] = df_final['FOB_UNIT_2024'].fillna(0)
df_final['FOB_UNIT_2025'] = df_final['FOB_UNIT_2025'].fillna(0)
df_final['CIF_UNIT_2023'] = df_final['CIF_UNIT_2023'].fillna(0)
df_final['CIF_UNIT_2024'] = df_final['CIF_UNIT_2024'].fillna(0)
df_final['CIF_UNIT_2025'] = df_final['CIF_UNIT_2025'].fillna(0)
df_final['CANTIDAD_2023'] = df_final['CANTIDAD_2023'].fillna(0)
df_final['CANTIDAD_2024'] = df_final['CANTIDAD_2024'].fillna(0)
df_final['CANTIDAD_2025'] = df_final['CANTIDAD_2025'].fillna(0)

df_final = df_final.drop(columns=['PRODUCTO_x', 'PRODUCTO_y', 'REPS'])

columnas = ['PRODUCTO', 'MODELO', 'CANTIDAD_2023', 'FOB_UNIT_2023', 'CIF_UNIT_2023', 'CANTIDAD_2024', 'FOB_UNIT_2024', 'CIF_UNIT_2024', 'CANTIDAD_2025', 'FOB_UNIT_2025', 'CIF_UNIT_2025']
df_final = df_final[columnas]

df_final

C:\Users\nicof\AppData\Local\Temp\ipykernel_3856\4209631929.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final['FOB_UNIT_2025'] = df_final['FOB_UNIT_2025'].fillna(0)
C:\Users\nicof\AppData\Local\Temp\ipykernel_3856\4209631929.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final['CIF_UNIT_2025'] = df_final['CIF_UNIT_2025'].fillna(0)
C:\Users\nicof\AppData\Local\Temp\ipykernel_3856\4209631929.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.in

,PRODUCTO,MODELO,CANTIDAD_2023,FOB_UNIT_2023,CIF_UNIT_2023,CANTIDAD_2024,FOB_UNIT_2024,CIF_UNIT_2024,CANTIDAD_2025,FOB_UNIT_2025,CIF_UNIT_2025
0,SCREW COMPRESSOR,0000-0087-13,0.0,0.0,0.0,0.0,0.00,0.000,1.0,283.84,337.98
1,SCREW COMPRESSOR,0000-0087-24,0.0,0.0,0.0,0.0,0.00,0.000,1.0,157.66,187.74
2,SCREW COMPRESSOR,0000-0087-49,0.0,0.0,0.0,0.0,0.00,0.000,1.0,201.95,240.47
3,SCREW COMPRESSOR,0000-0092-45,0.0,0.0,0.0,0.0,0.00,0.000,1.0,267.72,289.99
4,SCREW COMPRESSOR,0000-0505-24,0.0,0.0,0.0,0.0,0.00,0.000,1.0,111.14,135.65
...,...,...,...,...,...,...,...,...,...,...,...
1689,SECADOR,FX5 E0 ACUL 230V1PH60,0.0,0.0,0.0,2.0,1264.50,1544.500,0.0,0.00,0.00
1690,COMPRESOR TORNILLO,GA22VSDSFF A 150,0.0,0.0,0.0,2.0,20488.50,21110.710,0.0,0.00,0.00
1691,COMPRESOR TORNILLO,GA55VSD+FF A 175,0.0,0.0,0.0,1.0,28889.57,29766.910,0.0,0.00,0.00
1692,COMPRESOR TORNILLO,GA75P A 125,0.0,0.0,0.0,3.0,28825.78,29314.800,0.0,0.00,0.00


## Guardar cambios

In [171]:
df_final.to_excel('../docs/EMCOVELE_new_new.xlsx', index=False)